# 02 - Denoisers and deterministic samplers

Our network predicts velocity, while diffusion papers and the CIFAR experiment
often describe a denoised or clean-image estimate. Steering will be much easier
to understand once we can translate between those two languages. We then ask a
separate practical question: how accurately must a numerical solver follow the
learned field?


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


In [ ]:
model, losses, trained_now = load_or_train_model(device=device)
print("trained in this run:", trained_now)


## 1. Why can a velocity prediction be read as a denoiser?

Start with one exact training pair. Along the linear conditional path,

$$x_t=x_{noise}+t(x_{data}-x_{noise}).$$

Its pair-specific velocity is $u=x_{data}-x_{noise}$. Rearranging the path gives

$$x_{data}=x_t+(1-t)u.$$

This first identity is exact because the two endpoints are known while we build
a training example. During generation the network sees only $(x_t,t)$, not
those endpoints. Mean-squared-error training therefore learns the conditional
average

$$v^*(x_t,t)=\mathbb E[u\mid x_t,t].$$

Substituting that average into the exact identity gives

$$D^*(x_t,t)=x_t+(1-t)v^*(x_t,t)
=\mathbb E[x_{data}\mid x_t,t].$$

For the trained network we use the same conversion:

$$D_\theta(x_t,t)=x_t+(1-t)v_\theta(x_t,t).$$

The result is the best mean-squared-error clean estimate under the training
pair distribution. It is an average over plausible clean endpoints, not a
guarantee that the model has identified one hidden original. When several modes
are plausible, the average can sit between them; in images, such averaging can
look blurry.

Away from $t=1$, the conversion can also be inverted as
$v_\theta=(D_\theta-x_t)/(1-t)$. Near $t=1$ that division is poorly conditioned,
so code should not use it exactly at the endpoint. This dependence on the
chosen linear path is important: a different path has different conversion
factors.


In [ ]:
set_seed(SEED + 19)
visual_clean, visual_labels = sample_labeled_mixture(180, device=device)
visual_noise = sample_base(180, device=device)
visual_t_value = 0.35
visual_t = torch.full((180,), visual_t_value, device=device)
visual_xt = forward_noising(visual_clean, visual_t, visual_noise)
pair_velocity = visual_clean - visual_noise
with torch.no_grad():
    learned_velocity = model(visual_t, visual_xt)
    clean_estimate = denoised_from_velocity(visual_xt, visual_t, learned_velocity)

subset = torch.arange(0, 180, 6, device=device)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax in axes:
    ax.scatter(visual_xt[:, 0].cpu(), visual_xt[:, 1].cpu(), s=8, alpha=0.16, color="#777777")
    ax.set(aspect="equal", xlim=(-5, 5), ylim=(-5, 5))
    ax.set_xticks([])
    ax.set_yticks([])

axes[0].quiver(
    visual_xt[subset, 0].cpu(), visual_xt[subset, 1].cpu(),
    pair_velocity[subset, 0].cpu(), pair_velocity[subset, 1].cpu(),
    angles="xy", scale_units="xy", scale=3.0, width=0.004, color="#D55E00",
)
axes[0].set_title("Training target: endpoint pair is known")

axes[1].quiver(
    visual_xt[subset, 0].cpu(), visual_xt[subset, 1].cpu(),
    learned_velocity[subset, 0].cpu(), learned_velocity[subset, 1].cpu(),
    angles="xy", scale_units="xy", scale=3.0, width=0.004, color="#0072B2",
)
axes[1].set_title("Learned velocity: only (t, x_t) is known")

for index in subset:
    estimate_line = torch.stack([visual_xt[index], clean_estimate[index]]).cpu()
    axes[2].plot(estimate_line[:, 0], estimate_line[:, 1], color="#999999", alpha=0.45, lw=0.8)
axes[2].scatter(clean_estimate[:, 0].cpu(), clean_estimate[:, 1].cpu(), s=10, alpha=0.35, color="#0072B2", label="D_theta")
axes[2].scatter(visual_clean[:, 0].cpu(), visual_clean[:, 1].cpu(), s=14, alpha=0.25, facecolors="none", edgecolors="#D55E00", label="paired clean endpoint")
axes[2].set_title("D_theta = current state + remaining velocity")
axes[2].legend(frameon=False, fontsize=8, loc="upper left")
fig.suptitle(f"The velocity-to-denoiser conversion at t={visual_t_value:.2f}", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
set_seed(SEED + 20)
clean, labels = sample_labeled_mixture(450, device=device)
noise = sample_base(clean.shape[0], device=device)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, t_value in zip(axes, [0.15, 0.45, 0.75]):
    t = torch.full((clean.shape[0],), t_value, device=device)
    xt = forward_noising(clean, t, noise)
    with torch.no_grad():
        velocity = model(t, xt)
        denoised = denoised_from_velocity(xt, t, velocity)

    subset = torch.arange(0, clean.shape[0], 15, device=device)
    ax.scatter(xt[:, 0].cpu(), xt[:, 1].cpu(), s=7, alpha=0.18, color="#777777", label="current state")
    ax.scatter(denoised[:, 0].cpu(), denoised[:, 1].cpu(), s=8, alpha=0.35, color="#0072B2", label="denoised estimate")
    arrows = denoised[subset] - xt[subset]
    ax.quiver(
        xt[subset, 0].cpu(), xt[subset, 1].cpu(), arrows[:, 0].cpu(), arrows[:, 1].cpu(),
        angles="xy", scale_units="xy", scale=1, width=0.004, color="#D55E00", alpha=0.8,
    )
    ax.set_title(f"Denoiser view at t={t_value:.2f}")
    ax.set_aspect("equal")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
axes[0].legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()


## 2. Solver comparison

The model defines the arrows; the solver decides how to follow them. A fair
solver comparison changes only the solver and step count. It keeps the model,
initial noise tensor, and evaluation reference fixed.

Deterministic is conditional on the initial state: repeating the same
`x_noise` repeats the trajectory, while different initial noise points can
still produce a diverse batch of endpoints.

We use a 256-step RK4 trajectory as a numerical reference. This reference does
not remove model error; it only makes solver error small.


In [ ]:
set_seed(SEED + 21)
x0 = sample_base(1200, device=device)
target, _ = sample_labeled_mixture(1200, device=device)

reference_path, reference_seconds = timed_sample(base_velocity(model), x0, steps=256, method="rk4")
reference_endpoint = reference_path[-1]

rows = []
saved_paths = {}
for method in ["euler", "heun", "rk4"]:
    for steps in [8, 16, 32, 64]:
        path, seconds = timed_sample(base_velocity(model), x0, steps=steps, method=method)
        paired_rmse = torch.sqrt(torch.mean((path[-1] - reference_endpoint) ** 2)).item()
        rows.append({
            "solver": method,
            "steps": steps,
            "model evaluations": steps * {"euler": 1, "heun": 2, "rk4": 4}[method],
            "paired endpoint RMSE": paired_rmse,
            "Wasserstein distance to target": wasserstein_distance(path[-1], target.cpu()),
            "seconds": seconds,
        })
        if (method, steps) in {("euler", 8), ("heun", 16), ("rk4", 16)}:
            saved_paths[f"{method}, {steps} steps"] = path

solver_table = pd.DataFrame(rows)
display(solver_table.round(4))
print(f"reference RK4 runtime: {reference_seconds:.3f} s")


In [ ]:
plot_trajectory_comparison(saved_paths, n_lines=22)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for method, group in solver_table.groupby("solver"):
    ax.loglog(group["model evaluations"], group["paired endpoint RMSE"], marker="o", label=method)
ax.set(xlabel="model evaluations", ylabel="paired endpoint RMSE", title="Numerical accuracy at fixed initial noise")
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=False)
plt.show()


## 3. Two meanings that must remain separate

**Paired initial noise** is an experimental control. With a deterministic
sampler, the same initial tensor gives sample-by-sample correspondence between
two methods. Endpoint differences can then be attributed to the changed method.

**Noise alignment in NA-RFM** is a steering method. It uses the difference
between a target-class PCA denoiser and a full-data PCA denoiser during the
high-noise window. Notebook `03` implements that mechanism in 2D.

Using the same seed does not implement NA-RFM noise alignment. Conversely, a
noise-alignment experiment should still use paired initial noise for fair
evaluation.

**Questions**

1. Why does RK4 usually need fewer steps but more model evaluations per step?
2. Why is endpoint RMSE meaningful only because every solver starts from the exact same tensor?
3. Does a more accurate ODE solver guarantee a better trained vector field?
